In [64]:
import numpy as np
import pandas as pd

In [98]:
df = pd.read_csv("dirty_cafe_sales.csv")




# The diagnostic checklist -
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nMissing %:\n", (df.isnull().sum() / len(df) * 100).round(1))
print("\nDuplicate rows:", df.duplicated().sum())

# Check every object column for inconsistency
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} — unique values sample:")
    print(df[col].unique()[:15])

# This single loop reveals: mixed case, inconsistent 
# spelling, date format chaos, trailing whitespace — 


Shape: (10000, 8)

Dtypes:
 Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

Missing values:
 Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Missing %:
 Transaction ID       0.0
Item                 3.3
Quantity             1.4
Price Per Unit       1.8
Total Spent          1.7
Payment Method      25.8
Location            32.6
Transaction Date     1.6
dtype: float64

Duplicate rows: 0

Transaction ID — unique values sample:
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534' 'TXN_4717867' 'TXN_2064365'
 'TXN_2548360' 'TXN_3051279' 'TXN_7619095' 'TXN_9437049' 'TXN_8915701']

Item — un

# 1. Initial inspection and Structural Cleaning

In [66]:
df.shape

(10000, 8)

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [68]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [69]:
# Standardize column headers
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')


# 2. Handling Missing values - Find , DROP OR IMPUTE

In [70]:
# Finding:  percentage of null Values in each column
(df.isna().sum()/10000)*100


transaction_id       0.00
item                 3.33
quantity             1.38
price_per_unit       1.79
total_spent          1.73
payment_method      25.79
location            32.65
transaction_date     1.59
dtype: float64

In [71]:
# Replace 'UNKNOWN' and 'ERROR' string entries with actual NaN
df = df.replace(['','UNKNOWN', 'ERROR'], np.nan)


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9031 non-null   object
 2   quantity          9521 non-null   object
 3   price_per_unit    9467 non-null   object
 4   total_spent       9498 non-null   object
 5   payment_method    6822 non-null   object
 6   location          6039 non-null   object
 7   transaction_date  9540 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


# 3. Fixing Data Types

In [73]:
# Convert numeric columns, forcing remaining non-numbers to NaN
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'], errors='coerce')
df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce')

# Convert date column to actual Datetime objects
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')



## Impute AND DROP

In [86]:
# Recalculate total_spent where it's missing but we have quantity and price
df['total_spent'] = df['total_spent'].fillna(df['quantity'] * df['price_per_unit'])

# Fix Quantity where Total Spent and Price Per Unit exist
df['quantity'] = df['quantity'].fillna((df['total_spent'] / df['price_per_unit']).round())

# Fix Price Per Unit where Total Spent and Quantity exist
df['price_per_unit'] = df['price_per_unit'].fillna(df['total_spent'] / df['quantity'])

# Fill categorical missing values with a clean string baseline
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['location'] = df['location'].fillna('Unknown')

# Drop any row that still has unresolvable numeric gaps left over
df.dropna(subset=['item', 'transaction_date','quantity', 'price_per_unit', 'total_spent'], inplace=True)




(df.isna().sum()/10000)*100

transaction_id      0.0
item                0.0
quantity            0.0
price_per_unit      0.0
total_spent         0.0
payment_method      0.0
location            0.0
transaction_date    0.0
dtype: float64

# 4. String manipulation and de-duplications

In [87]:
# Clean up text columns
df['item'] = df['item'].str.strip().str.title()

# Deduplicate records based on the unique Transaction ID
df = df.drop_duplicates(subset=['transaction_id'])


In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8564 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    8564 non-null   object        
 1   item              8564 non-null   object        
 2   quantity          8564 non-null   float64       
 3   price_per_unit    8564 non-null   float64       
 4   total_spent       8564 non-null   float64       
 5   payment_method    8564 non-null   object        
 6   location          8564 non-null   object        
 7   transaction_date  8564 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 602.2+ KB


In [92]:
df.describe

<bound method NDFrame.describe of      transaction_id      item  quantity  price_per_unit  total_spent  \
0       TXN_1961373    Coffee       2.0             2.0          4.0   
1       TXN_4977031      Cake       4.0             3.0         12.0   
2       TXN_4271903    Cookie       4.0             1.0          4.0   
3       TXN_7034554     Salad       2.0             5.0         10.0   
4       TXN_3160411    Coffee       2.0             2.0          4.0   
...             ...       ...       ...             ...          ...   
9993    TXN_4766549  Smoothie       2.0             4.0          8.0   
9995    TXN_7672686    Coffee       2.0             2.0          4.0   
9997    TXN_5255387    Coffee       4.0             2.0          8.0   
9998    TXN_7695629    Cookie       3.0             1.0          3.0   
9999    TXN_6170729  Sandwich       3.0             4.0         12.0   

      payment_method  location transaction_date  
0        Credit Card  Takeaway       2023-09-08  
1

In [94]:
df['total_spent'].sum()

np.float64(76605.5)

In [95]:
df.to_csv('cleaned_cafe_sales.csv', index=False)